# 04 - Query status and position

Read fresh controller state, interpret status flags, and sample a bounded number of
position/status pairs. Queries configure the bridge when connecting but do not command motion.

Run cells from top to bottom in a fresh Python kernel. This notebook stands alone.
Install and launch as described in [the tutorial index](README.md).
Hardware access is disabled by default. Set your own eight-digit serial; no unit is
selected automatically. Close other notebooks' connections and other control software
before accessing the device. One connection owns the USB receive path.


In [ ]:
from dataclasses import asdict

from k10cr2 import K10CR2

SERIAL = ""  # Copy the serial from discovery and verify it on the mount's label.
CONNECT_TO_DEVICE = False  # Set True only when ready for USB access.


def configured_mount(**settings):
    # Construction validates the serial and timeouts but does not open USB.
    return K10CR2(SERIAL, **settings)

## Interpret a fresh snapshot

`busy` includes moving/jogging or homing. `homed` is the controller's reference flag.
`enabled` reports motor output state. A zero `motion_error` flag does not exclude the
general error bit (`0x40000000`), which the motion implementation also checks.
Position is a step counter converted to degrees, not an independent angle measurement.
Status and position use separate requests; they are not an atomic snapshot.


In [ ]:
if CONNECT_TO_DEVICE:
    with configured_mount() as mount:
        state = mount.status(timeout=3)
        position = mount.position(timeout=3)
        print(
            {
                **asdict(state),
                "busy": state.busy,
                "general_error": bool(state.raw_bits & 0x40000000),
                "raw_hex": f"0x{state.raw_bits:08X}",
            }
        )
        print(asdict(position))
else:
    print("Queries skipped; enter SERIAL and set CONNECT_TO_DEVICE = True.")

## Bounded polling

Five samples at approximately 0.5-second spacing, plus query latency. The timestamp is
elapsed software time. Polling does not reserve a second control path or issue streaming
commands. If any request fails, the context closes; the loop does not retry.


In [ ]:
import time

samples = []
if CONNECT_TO_DEVICE:
    with configured_mount() as mount:
        started = time.monotonic()
        for index in range(5):
            state = mount.status()
            position = mount.position()
            row = {
                "elapsed_s": time.monotonic() - started,
                "counts": position.counts,
                "degrees": position.degrees,
                "homed": state.homed,
                "busy": state.busy,
                "raw_bits": f"0x{state.raw_bits:08X}",
            }
            samples.append(row)
            print(row)
            if index < 4:
                time.sleep(0.5)
else:
    print("Polling skipped.")

After a query timeout or protocol failure, close and explicitly reconnect before
normal operations resume. Reconnecting does not establish a new home reference.

Next: [05 - Settings and units](05_settings_and_units.ipynb).
